## LangGraph zRAG agent (based on Custom RAG agent)
https://docs.langchain.com/oss/python/langgraph/agentic-rag

### 2. Create a zRAG retriever tool

In [ ]:
from zrag_util import ZRAGRetriever, format_documents

query_file = "zrag_retriever_data.jsonl"
retriever = ZRAGRetriever.from_json(query_file)
# retriever.invoke("IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？")

In [ ]:
from langchain.tools import tool

# OK for gpt-oss-128b, Not OK for granite-4.1-30b
# """
# Search information from IBM Z documents.
# Target areas: z/OS, CICS, IMS, DB2, MQ, JCL, VSAM, RACF, TSO/ISPF, USS, DevOps, system programming, batch processing, mainframe technologies, Dependency Based Build (DBB) and git.
# """

# OK for both
# """
# IBM Zのドキュメントから、情報を検索します。
# 対象分野：z/OS, CICS, IMS, DB2, MQ, JCL, VSAM, RACF, TSO/ISPF, USS, DevOps, system programming, batch processing, mainframe technologies, Dependency Based Build (DBB) and git.
# """

@tool
def zrag_retriever(query: str) -> str:
    """
    IBM Zのドキュメントから、情報を検索します。
    対象分野：z/OS, CICS, IMS, DB2, MQ, JCL, VSAM, RACF, TSO/ISPF, USS, DevOps, system programming, batch processing, mainframe technologies, Dependency Based Build (DBB) and git.
    """
    docs = retriever.invoke(query)
    return format_documents(docs)

retriever_tool = zrag_retriever
# retriever_tool.invoke({"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"})

### 3. Generate query node

#### ノード名 `generate_query_or_respond`
#### 入力
```python
{
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "質問",
            }
        ]
    )
}
```
#### 出力
AIMessage
```python
{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'AIMessage'], 'kwargs': {'content': '', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 90, 'prompt_tokens': 139, 'total_tokens': 229, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-a183d9a0-0966-4216-9be8-f08bc3184423', 'finish_reason': 'tool_calls', 'logprobs': None}, 'type': 'ai', 'id': 'lc_run--019e456a-7a4d-7b50-bba6-e139c1e23b65-0', 'tool_calls': [{'name': 'zrag_retriever', 'args': {'query': 'IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？'}, 'id': 'chatcmpl-tool-b5b84c5010f81a30', 'type': 'tool_call'}], 'usage_metadata': {'input_tokens': 139, 'output_tokens': 90, 'total_tokens': 229, 'input_token_details': {}, 'output_token_details': {}}, 'invalid_tool_calls': []}}
```
or
AIMessage
```python
{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'AIMessage'], 'kwargs': {'content': 'Hello! How can I help you today?', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 36, 'prompt_tokens': 128, 'total_tokens': 164, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-1a105271-1f0a-4d7e-9f85-9d2c53dedd37', 'finish_reason': 'stop', 'logprobs': None}, 'type': 'ai', 'id': 'lc_run--019e4575-3630-7562-88dd-21981f394621-0', 'usage_metadata': {'input_tokens': 128, 'output_tokens': 36, 'total_tokens': 164, 'input_token_details': {}, 'output_token_details': {}}, 'tool_calls': [], 'invalid_tool_calls': []}}
```

In [ ]:
import os
from langchain_openai import ChatOpenAI

response_model = ChatOpenAI(
    # model="mistralai/Mistral-Large-3-675B-Instruct-2512-NVFP4",
    # base_url="https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/mistral-large-3-675b-2512-fp4/v1",
    # model="openai/gpt-oss-120b",
    # base_url="https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/gpt-oss-120b/v1",
    model="ibm-granite/granite-4.1-30b",
    base_url="https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/granite-4-1-30b/v1",
    api_key="DUMMY", # type: ignore
    temperature=0,
    default_headers={
        "RITS_API_KEY": os.getenv("RITS_API_KEY"),
    }, # type: ignore
)
# print(response_model)

In [ ]:
from langgraph.graph import MessagesState

def print_messages(state: MessagesState, header: str | None = None, print_: bool = False):
    if not print_:
        return

    if header:
        print(f"XXX {header} XXX")
    for message in state["messages"]:
        message.pretty_print()
    print("\n--------------------------------------------------------------------------------\n\n")

In [ ]:
from langgraph.graph import MessagesState

def generate_query_or_respond(state: MessagesState):
    """Call the model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply respond to the user.
    """
    print_messages(state, "generate_query_or_respond")
    response = (
        response_model
        .bind_tools([retriever_tool]).invoke(state["messages"])
    )
    # @@@ahoaho XXX
    # NOTE: update the tool call's "query" arg with the latest (original or rewritten) user query
    # if getattr(response, "tool_calls", None):
    if response.tool_calls:
        for message in reversed(state["messages"]):
            from langchain.messages import HumanMessage
            if isinstance(message, HumanMessage):
                query = message.content
                # tool_call = getattr(response, "tool_calls")[0]
                tool_call = response.tool_calls[0]
                # print(f"XXX original: {tool_call}")
                tool_call["args"]["query"] = query
                # print(f"XXX updated: {tool_call}")
                break
    return {"messages": [response]}

#### Test `generate_query_or_respond` node

In [ ]:
from langchain_core.messages import convert_to_messages

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "hello!",
            }
        ]
    )
}

last_message = generate_query_or_respond(input)["messages"][-1] # type: ignore
# print(last_message.to_json())
last_message.pretty_print()

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            }
        ]
    )
}

last_message = generate_query_or_respond(input)["messages"][-1] # type: ignore
# print(last_message.to_json())
last_message.pretty_print()

### 4. Grade documents path

#### パス名 `grade_document`
#### 入力
```python
{
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "zrag_retriever",
                        "args": {"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "ニャー",
                "tool_call_id": "1",
            },
        ]
    )
}
```
#### 出力
```python
"rewrite_question"
```
or
```python
"generate_answer"
```

In [ ]:
import os
from langchain_openai import ChatOpenAI

grader_model = ChatOpenAI(
    # model="mistralai/Mistral-Large-3-675B-Instruct-2512-NVFP4",
    # base_url="https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/mistral-large-3-675b-2512-fp4/v1",
    model="openai/gpt-oss-120b",
    base_url="https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/gpt-oss-120b/v1",
    api_key="DUMMY", # type: ignore
    temperature=0,
    default_headers={
        "RITS_API_KEY": os.getenv("RITS_API_KEY"),
    }, # type: ignore
)
# print(grader_model)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

GRADE_PROMPT = (
    "You are a grader assessing relevance of a retrieved document to a user question. \n "
    "Here is the retrieved document: \n\n {context} \n\n"
    "Here is the user question: {question} \n"
    "If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n"
    "Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."
)

class GradeDocuments(BaseModel):
    """Grade documents using a binary score for relevance check."""

    binary_score: str = Field(
        description="Relevance score: 'yes' if relevant, or 'no' if not relevant"
    )

# NOTE: grade_documents is a path (conditional edge)
def grade_documents(
    state: MessagesState,
) -> Literal["generate_answer", "rewrite_question"]:
    """Determine whether the retrieved documents are relevant to the question."""
    print_messages(state, "grade_documents")
    question = state["messages"][0].content
    context = state["messages"][-1].content

    prompt = GRADE_PROMPT.format(question=question, context=context)
    response = (
        grader_model
        .with_structured_output(GradeDocuments).invoke(
            [{"role": "user", "content": prompt}]
        )
    )
    score = response.binary_score # type: ignore

    if score == "yes":
        return "generate_answer"
    else:
        return "rewrite_question"

#### Test `grade_documents` path

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "zrag_retriever",
                        "args": {"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "ニャー",
                "tool_call_id": "1",
            },
        ]
    )
}

grade_documents(input) # type: ignore

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "zrag_retriever",
                        "args": {"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "[Rank 1] Title: IBM Information Management System (IMS) IBM Information Management System (IMS) IMS 15.6 - System definition - PWFI= parameter for procedures\nURL: https://www.ibm.com/docs/en/ims/15.6.0?topic=parameters-pwfi-parameter-procedures\nScore: 0.412949800491333\nContent: IMS 15.6 - System definition - PWFI= parameter for procedures\nPWFI= parameter for procedures   \n Use the PWFI= parameter in procedure to specify the pseudo-wait-for-input (PWFI) parameter. Valid values are Y or N. N is the default. \n     Y   Activates pseudo wait-for-input (PWFI).  If the application program issues a Get Unique (GU) call to get a message from the IMS message queue, MODE=SINGLE is specified on the TRANSACT macro and no message is available, IMS checks for other work for this region.  If no other work is available, instead of returning status QC to the application program, IMS enqueues this region on scheduler subqueue 6 indicating that it is a PWFI region. If the next message is for the transaction scheduled in this MPP region, this region is dequeued from subqueue 6 and posted. IMS then returns the new message to the application program. This eliminates the rescheduling of IMS resources. If the processing limit (PROCLIM) for the transaction is reached, the application receives a QC status and is expected to terminate. After the application program terminates, it is reloaded on the subsequent schedule. \n N   Disables PWFI. If the program issues a Get Unique (GU) call to get a message from the IMS message queue but no message is available, the application program receives status QC.\n#\n\n\n[Rank 2] Title: IBM Information Management System (IMS) IBM Information Management System (IMS) IMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nURL: https://www.ibm.com/docs/en/ims/15.6.0?topic=dis-pseudo-wfi-option-mpp-regions-in-dbdc-dcctl-environments\nScore: 0.17378094792366028\nContent: IMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nPseudo WFI option for MPP regions in DB/DC and DCCTL environments   \n The pseudo WFI (pseudo wait-for-input) option allows an MPP region to remain scheduled until another input message appears. With pseudo WFI, unnecessary application program termination and rescheduling can be eliminated. \n Normally, if an MPP region is scheduled for a transaction and no more messages for that transaction exist, the application program terminates. Frequently, another message appears for the same transaction after the program is terminated. Processor usage is increased because of unnecessary termination and rescheduling of that application program. \n Pseudo WFI is specified with the  PWFI=  parameter on the MPP region startup procedure. When  PWFI= Y  is specified, the processing limit count is greater than 0, the PSB is not allocated with the dynamic PSB option (DOPT) and no more messages are queued for the current  MODE= SNGL  transaction, IMS checks for other work for the region to process. If no other work is available, the region waits until another input message appears. This is  wait-for-input  mode. \n When the next input message is for the currently scheduled transaction, the message is returned to the application program with a status code of  blank-blank . \n When the next message is not for the currently scheduled transaction, termination and rescheduling occur. \n In certain circumstances, regions that are in wait-for-input mode will be posted and a QC status code is returned to the application program. These circumstances include:   Commands that involve stopping, starting, locking, unlocking, or purging \n Commands that involve assigning a database, region, transaction, or class \n The  UPDATE PGM START(REFRESH)  command\nIMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nCommands that involve assigning a database, region, transaction, or class \n The  UPDATE PGM START(REFRESH)  command \n Note:  The  UPDATE PGM START(REFRESH)  command is not supported for MPP regions where the program is loaded by the DFSMPLxx PROCLIB member.   When application program changes are made in IMS.PGMLIB, you can issue the  UPDATE PGM START(REFRESH)  command to post the MPP PWFI regions so that a refreshed copy of the application program can be obtained when the program gets scheduled again. By using this command, you do not have to find all of the regions that the program is scheduled in and to stop those regions manually. The  UPDATE PGM START(REFRESH)  command is supported for programs scheduled in MPP PWFI regions that have the program scheduled in them and the program is not preloaded by the DFSMPLxx PROCLIB member, . \n \n     The following circumstances also cause similar problems:   An intent conflict scheduling failure might occur as a consequence of either a local or global load balancing algorithm decision. This causes all PWFI regions to be posted and QC status codes to be returned to the application programs.  The global load balancing algorithm is a feature of Sysplex Serial Program Manager (SSPM). \n \n A resource enters the OLC PREPARE phase. This causes all PWFI regions to be posted and a QC status code to be returned to the application programs. \n A program or transaction is stopped as a consequence of a dependent region abend. This causes all PWFI regions to be posted and QC status codes to be returned to the application programs that belong to regions with stopped programs or transactions. \n     Regions that cannot be scheduled because of a lack of pool space can also post regions currently in pseudo WFI in an attempt to terminate them. This frees pool space so that the failing region can schedule.\n#\n\n\n[Rank 3] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Accessing the WSim Test Manager\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=environment-accessing-wsim-test-manager\nScore: 0.05096757411956787\nContent: Accessing the WSim Test Manager   \n After the WSim Test Manager is installed, decide how you will provide access to the tool. An entry point can be added to the main ISPF menu that allows access to anyone on the system, or individual access can be provided so that only those users that are defined to WTM can invoke it. \n For both situations, make a copy of the WTM exec,  WSIMTM , in the WSIM.SITPEXEC data set. For global access, make a copy of the  WSIMTM  exec in a data set that will be concatenated under the SYSPROC DD statement in the TSO logon procedure. For individual access, make a copy of the  WSIMTM  exec into a CLIST (sequential data set) using your own high level qualifier (for example, userid.WSIMTM.CLIST). Then modify the copy of the  WSIMTM  exec or CLIST by adding the WSim Version 1 Release 1.0.0 data set names in the  \"update data set names\"  section. Because WTM invokes the WSim utilities, WTM does not work if the WSim data set names are missing or incorrect. \n The  WSIMTM  exec concatenates the following data sets: \n \n Note:  Data set names for WTM might be different from those in the following list; for an SMP install, the WTM data set names are defined by the user.   \n WSIM.SITPPNL   Data set concatenated to the ISPPLIB DD statement \n WSIM.SITPMSG   Data set concatenated to the ISPMLIB DD statement \n WSIM.SITPEXEC   Data set concatenated to the SYSEXEC DD statement \n WSIM.SITPTBL   Data set concatenated to the ISPTLIB statement \n userid.WTMUSER.SKELS   Data set concatenated to the ISPSLIB DD statement (the prefix userid does not need to be specified in the WSIMTM exec for WTMUSER.SKELS)\nAccessing the WSim Test Manager\nuserid.WTMUSER.SKELS   Data set concatenated to the ISPSLIB DD statement (the prefix userid does not need to be specified in the WSIMTM exec for WTMUSER.SKELS) \n   \n Note:  When using the WSim Test Manager, the TSO profile user characteristic, PREFIX, is set to the user ID. This ensures that the user ID is added as the first qualifier for all non-fully qualified data set names. The TSO profile user characteristics, WTPMSG and MSGID, are set to show MVS™ messages and terminal message IDs for debugging purposes. The PREFIX, WTPMSG, and MSGID settings are restored to the original values when the WSim Test Manager ends normally.   The  WSIMTM  exec or CLIST cannot be executed until the new user is set up (as described in  Setting up a new user ). \n         \n Global access to WTM   \n \n Individual access to WTM\n#\n\n\n[Rank 4] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Running WSim with the WSim/ISPF Interface\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=utilities-running-wsim-wsimispf-interface\nScore: 0.047599270939826965\nContent: Running  WSim  with the  WSim/ISPF Interface \n   \n The  WSim/ISPF Interface  is a CUA-compliant operator interface that lets you  start the  WSim  utilities.   To use the  WSim/ISPF Interface , you need to run  WSim  under MVS™ with TSO/E and ISPF.  Refer to  WSim User's Guide  for information about setting up your environment to run the  WSim/ISPF Interface .  \n \n You can access the following  WSim  functions with the  WSim/ISPF Interface :   \n STL Translator \n Preprocessor and ITPSYSIN \n Interactive Data Capture (IDC) Utility \n Log Script Generator Utility \n Script Generator Utility \n SNA 3270 Reformatter Utility \n \n WSim  simulation runs \n Response Time Utility \n Log Compare Utility \n Loglist Utility \n   You can get help for the  WSim/ISPF Interface  at any time by pressing F1. For more information, see  Getting help from the WSim/ISPF Interface .         \n Invoking the WSim/ISPF Interface   \n \n Getting help from the WSim/ISPF Interface   \n \n Navigating the panels   \n \n Entering data on panel fields   \n \n Using function keys\n#\n\n\n[Rank 5] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Operation\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=guide-operation\nScore: 0.03493978828191757\nContent: Operation           \n Introduction to WSim operation   \n \n Running WSim   \n \n Using operator commands   \n \n Using operator reports   \n \n Controlling message logging   \n \n Using the Display Monitor Facility   \n \n Isolating problems   \n \n Specifying operator commands\n#\n",
                "tool_call_id": "1",
            },
        ]
    )
}

grade_documents(input) # type: ignore

### 5. Rewrite question node

#### ノード名 `rewrite_question`
#### 入力
```python
{
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "zrag_retriever",
                        "args": {"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "ニャー",
                "tool_call_id": "1",
            },
        ]
    )
}
```
### 出力
HumanMessage
```python
{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'HumanMessage'], 'kwargs': {'content': 'IMSの待機入力（WFI）トランザクションを使用することで、どのような具体的な利点や目的が達成されるのか、その主な目的や使用ケースについて詳しく説明してください。', 'type': 'human'}}
```

In [ ]:
from langchain.messages import HumanMessage

# "Formulate an improved question:"
REWRITE_PROMPT = (
    "Look at the input and try to reason about the underlying semantic intent / meaning.\n"
    "Here is the initial question:"
    "\n ------- \n"
    "{question}"
    "\n ------- \n"
    # @@@ahoaho XXX
    # "Formulate an improved question:"
    "Formulate an improved question in the same language:"
)

# NOTE: Node
def rewrite_question(state: MessagesState):
    """Rewrite the original user question."""
    print_messages(state, "rewrite_question")
    messages = state["messages"]
    question = messages[0].content
    prompt = REWRITE_PROMPT.format(question=question)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [HumanMessage(content=response.content)]}

#### Test `rewrite_question` node

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "zrag_retriever",
                        "args": {"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "ニャー",
                "tool_call_id": "1",
            },
        ]
    )
}

response = rewrite_question(input) # type: ignore
last_message = response["messages"][-1]
# print(last_message.to_json())
last_message.pretty_print()

### 6. Generate an answer node

#### ノード名 `generate_answer`
#### 入力
```python
{
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "zrag_retriever",
                        "args": {"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "[Rank 1] Title: IBM Information Management System (IMS) IBM Information Management System (IMS) IMS 15.6 - System definition - PWFI= parameter for procedures\nURL: https://www.ibm.com/docs/en/ims/15.6.0?topic=parameters-pwfi-parameter-procedures\nScore: 0.412949800491333\nContent: IMS 15.6 - System definition - PWFI= parameter for procedures\nPWFI= parameter for procedures   \n Use the PWFI= parameter in procedure to specify the pseudo-wait-for-input (PWFI) parameter. Valid values are Y or N. N is the default. \n     Y   Activates pseudo wait-for-input (PWFI).  If the application program issues a Get Unique (GU) call to get a message from the IMS message queue, MODE=SINGLE is specified on the TRANSACT macro and no message is available, IMS checks for other work for this region.  If no other work is available, instead of returning status QC to the application program, IMS enqueues this region on scheduler subqueue 6 indicating that it is a PWFI region. If the next message is for the transaction scheduled in this MPP region, this region is dequeued from subqueue 6 and posted. IMS then returns the new message to the application program. This eliminates the rescheduling of IMS resources. If the processing limit (PROCLIM) for the transaction is reached, the application receives a QC status and is expected to terminate. After the application program terminates, it is reloaded on the subsequent schedule. \n N   Disables PWFI. If the program issues a Get Unique (GU) call to get a message from the IMS message queue but no message is available, the application program receives status QC.\n#\n\n\n[Rank 2] Title: IBM Information Management System (IMS) IBM Information Management System (IMS) IMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nURL: https://www.ibm.com/docs/en/ims/15.6.0?topic=dis-pseudo-wfi-option-mpp-regions-in-dbdc-dcctl-environments\nScore: 0.17378094792366028\nContent: IMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nPseudo WFI option for MPP regions in DB/DC and DCCTL environments   \n The pseudo WFI (pseudo wait-for-input) option allows an MPP region to remain scheduled until another input message appears. With pseudo WFI, unnecessary application program termination and rescheduling can be eliminated. \n Normally, if an MPP region is scheduled for a transaction and no more messages for that transaction exist, the application program terminates. Frequently, another message appears for the same transaction after the program is terminated. Processor usage is increased because of unnecessary termination and rescheduling of that application program. \n Pseudo WFI is specified with the  PWFI=  parameter on the MPP region startup procedure. When  PWFI= Y  is specified, the processing limit count is greater than 0, the PSB is not allocated with the dynamic PSB option (DOPT) and no more messages are queued for the current  MODE= SNGL  transaction, IMS checks for other work for the region to process. If no other work is available, the region waits until another input message appears. This is  wait-for-input  mode. \n When the next input message is for the currently scheduled transaction, the message is returned to the application program with a status code of  blank-blank . \n When the next message is not for the currently scheduled transaction, termination and rescheduling occur. \n In certain circumstances, regions that are in wait-for-input mode will be posted and a QC status code is returned to the application program. These circumstances include:   Commands that involve stopping, starting, locking, unlocking, or purging \n Commands that involve assigning a database, region, transaction, or class \n The  UPDATE PGM START(REFRESH)  command\nIMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nCommands that involve assigning a database, region, transaction, or class \n The  UPDATE PGM START(REFRESH)  command \n Note:  The  UPDATE PGM START(REFRESH)  command is not supported for MPP regions where the program is loaded by the DFSMPLxx PROCLIB member.   When application program changes are made in IMS.PGMLIB, you can issue the  UPDATE PGM START(REFRESH)  command to post the MPP PWFI regions so that a refreshed copy of the application program can be obtained when the program gets scheduled again. By using this command, you do not have to find all of the regions that the program is scheduled in and to stop those regions manually. The  UPDATE PGM START(REFRESH)  command is supported for programs scheduled in MPP PWFI regions that have the program scheduled in them and the program is not preloaded by the DFSMPLxx PROCLIB member, . \n \n     The following circumstances also cause similar problems:   An intent conflict scheduling failure might occur as a consequence of either a local or global load balancing algorithm decision. This causes all PWFI regions to be posted and QC status codes to be returned to the application programs.  The global load balancing algorithm is a feature of Sysplex Serial Program Manager (SSPM). \n \n A resource enters the OLC PREPARE phase. This causes all PWFI regions to be posted and a QC status code to be returned to the application programs. \n A program or transaction is stopped as a consequence of a dependent region abend. This causes all PWFI regions to be posted and QC status codes to be returned to the application programs that belong to regions with stopped programs or transactions. \n     Regions that cannot be scheduled because of a lack of pool space can also post regions currently in pseudo WFI in an attempt to terminate them. This frees pool space so that the failing region can schedule.\n#\n\n\n[Rank 3] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Accessing the WSim Test Manager\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=environment-accessing-wsim-test-manager\nScore: 0.05096757411956787\nContent: Accessing the WSim Test Manager   \n After the WSim Test Manager is installed, decide how you will provide access to the tool. An entry point can be added to the main ISPF menu that allows access to anyone on the system, or individual access can be provided so that only those users that are defined to WTM can invoke it. \n For both situations, make a copy of the WTM exec,  WSIMTM , in the WSIM.SITPEXEC data set. For global access, make a copy of the  WSIMTM  exec in a data set that will be concatenated under the SYSPROC DD statement in the TSO logon procedure. For individual access, make a copy of the  WSIMTM  exec into a CLIST (sequential data set) using your own high level qualifier (for example, userid.WSIMTM.CLIST). Then modify the copy of the  WSIMTM  exec or CLIST by adding the WSim Version 1 Release 1.0.0 data set names in the  \"update data set names\"  section. Because WTM invokes the WSim utilities, WTM does not work if the WSim data set names are missing or incorrect. \n The  WSIMTM  exec concatenates the following data sets: \n \n Note:  Data set names for WTM might be different from those in the following list; for an SMP install, the WTM data set names are defined by the user.   \n WSIM.SITPPNL   Data set concatenated to the ISPPLIB DD statement \n WSIM.SITPMSG   Data set concatenated to the ISPMLIB DD statement \n WSIM.SITPEXEC   Data set concatenated to the SYSEXEC DD statement \n WSIM.SITPTBL   Data set concatenated to the ISPTLIB statement \n userid.WTMUSER.SKELS   Data set concatenated to the ISPSLIB DD statement (the prefix userid does not need to be specified in the WSIMTM exec for WTMUSER.SKELS)\nAccessing the WSim Test Manager\nuserid.WTMUSER.SKELS   Data set concatenated to the ISPSLIB DD statement (the prefix userid does not need to be specified in the WSIMTM exec for WTMUSER.SKELS) \n   \n Note:  When using the WSim Test Manager, the TSO profile user characteristic, PREFIX, is set to the user ID. This ensures that the user ID is added as the first qualifier for all non-fully qualified data set names. The TSO profile user characteristics, WTPMSG and MSGID, are set to show MVS™ messages and terminal message IDs for debugging purposes. The PREFIX, WTPMSG, and MSGID settings are restored to the original values when the WSim Test Manager ends normally.   The  WSIMTM  exec or CLIST cannot be executed until the new user is set up (as described in  Setting up a new user ). \n         \n Global access to WTM   \n \n Individual access to WTM\n#\n\n\n[Rank 4] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Running WSim with the WSim/ISPF Interface\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=utilities-running-wsim-wsimispf-interface\nScore: 0.047599270939826965\nContent: Running  WSim  with the  WSim/ISPF Interface \n   \n The  WSim/ISPF Interface  is a CUA-compliant operator interface that lets you  start the  WSim  utilities.   To use the  WSim/ISPF Interface , you need to run  WSim  under MVS™ with TSO/E and ISPF.  Refer to  WSim User's Guide  for information about setting up your environment to run the  WSim/ISPF Interface .  \n \n You can access the following  WSim  functions with the  WSim/ISPF Interface :   \n STL Translator \n Preprocessor and ITPSYSIN \n Interactive Data Capture (IDC) Utility \n Log Script Generator Utility \n Script Generator Utility \n SNA 3270 Reformatter Utility \n \n WSim  simulation runs \n Response Time Utility \n Log Compare Utility \n Loglist Utility \n   You can get help for the  WSim/ISPF Interface  at any time by pressing F1. For more information, see  Getting help from the WSim/ISPF Interface .         \n Invoking the WSim/ISPF Interface   \n \n Getting help from the WSim/ISPF Interface   \n \n Navigating the panels   \n \n Entering data on panel fields   \n \n Using function keys\n#\n\n\n[Rank 5] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Operation\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=guide-operation\nScore: 0.03493978828191757\nContent: Operation           \n Introduction to WSim operation   \n \n Running WSim   \n \n Using operator commands   \n \n Using operator reports   \n \n Controlling message logging   \n \n Using the Display Monitor Facility   \n \n Isolating problems   \n \n Specifying operator commands\n#\n",
                "tool_call_id": "1",
            },
        ]
    )
}
```
#### 出力
AIMessage
```python
{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'messages', 'AIMessage'], 'kwargs': {'content': 'IMSの待機入力（WFI）トランザクションを使用する主な目的は、MPP領域が別の入力メッセージが到着するまでスケジュールされたままになることを可能にし、不要なアプリケーションプログラムの終了と再スケジューリングを排除することです。これにより、プロセッサ使用率が向上します。PWFI=パラメータを使用してこの機能を有効にします。', 'additional_kwargs': {'refusal': None}, 'response_metadata': {'token_usage': {'completion_tokens': 147, 'prompt_tokens': 151, 'total_tokens': 298, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-4a495975-c237-4103-a5b3-085e00857aa2', 'finish_reason': 'stop', 'logprobs': None}, 'type': 'ai', 'id': 'lc_run--019e4581-b7b7-7e03-92f6-bd13fa9f9ef6-0', 'usage_metadata': {'input_tokens': 151, 'output_tokens': 147, 'total_tokens': 298, 'input_token_details': {}, 'output_token_details': {}}, 'tool_calls': [], 'invalid_tool_calls': []}}
```

In [ ]:
GENERATE_PROMPT = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \n"
    "Context: {context}"
)

# NOTE: Node
def generate_answer(state: MessagesState):
    """Generate an answer."""
    print_messages(state, "generate_answer")
    question = state["messages"][0].content
    context = state["messages"][-1].content
    prompt = GENERATE_PROMPT.format(question=question, context=context)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [response]}

#### Test `generate_answer` node

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            },
            {
                "role": "assistant",
                "content": "",
                "tool_calls": [
                    {
                        "id": "1",
                        "name": "zrag_retriever",
                        "args": {"query": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？"},
                    }
                ],
            },
            {
                "role": "tool",
                "content": "[Rank 1] Title: IBM Information Management System (IMS) IBM Information Management System (IMS) IMS 15.6 - System definition - PWFI= parameter for procedures\nURL: https://www.ibm.com/docs/en/ims/15.6.0?topic=parameters-pwfi-parameter-procedures\nScore: 0.412949800491333\nContent: IMS 15.6 - System definition - PWFI= parameter for procedures\nPWFI= parameter for procedures   \n Use the PWFI= parameter in procedure to specify the pseudo-wait-for-input (PWFI) parameter. Valid values are Y or N. N is the default. \n     Y   Activates pseudo wait-for-input (PWFI).  If the application program issues a Get Unique (GU) call to get a message from the IMS message queue, MODE=SINGLE is specified on the TRANSACT macro and no message is available, IMS checks for other work for this region.  If no other work is available, instead of returning status QC to the application program, IMS enqueues this region on scheduler subqueue 6 indicating that it is a PWFI region. If the next message is for the transaction scheduled in this MPP region, this region is dequeued from subqueue 6 and posted. IMS then returns the new message to the application program. This eliminates the rescheduling of IMS resources. If the processing limit (PROCLIM) for the transaction is reached, the application receives a QC status and is expected to terminate. After the application program terminates, it is reloaded on the subsequent schedule. \n N   Disables PWFI. If the program issues a Get Unique (GU) call to get a message from the IMS message queue but no message is available, the application program receives status QC.\n#\n\n\n[Rank 2] Title: IBM Information Management System (IMS) IBM Information Management System (IMS) IMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nURL: https://www.ibm.com/docs/en/ims/15.6.0?topic=dis-pseudo-wfi-option-mpp-regions-in-dbdc-dcctl-environments\nScore: 0.17378094792366028\nContent: IMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nPseudo WFI option for MPP regions in DB/DC and DCCTL environments   \n The pseudo WFI (pseudo wait-for-input) option allows an MPP region to remain scheduled until another input message appears. With pseudo WFI, unnecessary application program termination and rescheduling can be eliminated. \n Normally, if an MPP region is scheduled for a transaction and no more messages for that transaction exist, the application program terminates. Frequently, another message appears for the same transaction after the program is terminated. Processor usage is increased because of unnecessary termination and rescheduling of that application program. \n Pseudo WFI is specified with the  PWFI=  parameter on the MPP region startup procedure. When  PWFI= Y  is specified, the processing limit count is greater than 0, the PSB is not allocated with the dynamic PSB option (DOPT) and no more messages are queued for the current  MODE= SNGL  transaction, IMS checks for other work for the region to process. If no other work is available, the region waits until another input message appears. This is  wait-for-input  mode. \n When the next input message is for the currently scheduled transaction, the message is returned to the application program with a status code of  blank-blank . \n When the next message is not for the currently scheduled transaction, termination and rescheduling occur. \n In certain circumstances, regions that are in wait-for-input mode will be posted and a QC status code is returned to the application program. These circumstances include:   Commands that involve stopping, starting, locking, unlocking, or purging \n Commands that involve assigning a database, region, transaction, or class \n The  UPDATE PGM START(REFRESH)  command\nIMS 15.6 - System definition - Pseudo WFI option for MPP regions in DB/DC and DCCTL environments\nCommands that involve assigning a database, region, transaction, or class \n The  UPDATE PGM START(REFRESH)  command \n Note:  The  UPDATE PGM START(REFRESH)  command is not supported for MPP regions where the program is loaded by the DFSMPLxx PROCLIB member.   When application program changes are made in IMS.PGMLIB, you can issue the  UPDATE PGM START(REFRESH)  command to post the MPP PWFI regions so that a refreshed copy of the application program can be obtained when the program gets scheduled again. By using this command, you do not have to find all of the regions that the program is scheduled in and to stop those regions manually. The  UPDATE PGM START(REFRESH)  command is supported for programs scheduled in MPP PWFI regions that have the program scheduled in them and the program is not preloaded by the DFSMPLxx PROCLIB member, . \n \n     The following circumstances also cause similar problems:   An intent conflict scheduling failure might occur as a consequence of either a local or global load balancing algorithm decision. This causes all PWFI regions to be posted and QC status codes to be returned to the application programs.  The global load balancing algorithm is a feature of Sysplex Serial Program Manager (SSPM). \n \n A resource enters the OLC PREPARE phase. This causes all PWFI regions to be posted and a QC status code to be returned to the application programs. \n A program or transaction is stopped as a consequence of a dependent region abend. This causes all PWFI regions to be posted and QC status codes to be returned to the application programs that belong to regions with stopped programs or transactions. \n     Regions that cannot be scheduled because of a lack of pool space can also post regions currently in pseudo WFI in an attempt to terminate them. This frees pool space so that the failing region can schedule.\n#\n\n\n[Rank 3] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Accessing the WSim Test Manager\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=environment-accessing-wsim-test-manager\nScore: 0.05096757411956787\nContent: Accessing the WSim Test Manager   \n After the WSim Test Manager is installed, decide how you will provide access to the tool. An entry point can be added to the main ISPF menu that allows access to anyone on the system, or individual access can be provided so that only those users that are defined to WTM can invoke it. \n For both situations, make a copy of the WTM exec,  WSIMTM , in the WSIM.SITPEXEC data set. For global access, make a copy of the  WSIMTM  exec in a data set that will be concatenated under the SYSPROC DD statement in the TSO logon procedure. For individual access, make a copy of the  WSIMTM  exec into a CLIST (sequential data set) using your own high level qualifier (for example, userid.WSIMTM.CLIST). Then modify the copy of the  WSIMTM  exec or CLIST by adding the WSim Version 1 Release 1.0.0 data set names in the  \"update data set names\"  section. Because WTM invokes the WSim utilities, WTM does not work if the WSim data set names are missing or incorrect. \n The  WSIMTM  exec concatenates the following data sets: \n \n Note:  Data set names for WTM might be different from those in the following list; for an SMP install, the WTM data set names are defined by the user.   \n WSIM.SITPPNL   Data set concatenated to the ISPPLIB DD statement \n WSIM.SITPMSG   Data set concatenated to the ISPMLIB DD statement \n WSIM.SITPEXEC   Data set concatenated to the SYSEXEC DD statement \n WSIM.SITPTBL   Data set concatenated to the ISPTLIB statement \n userid.WTMUSER.SKELS   Data set concatenated to the ISPSLIB DD statement (the prefix userid does not need to be specified in the WSIMTM exec for WTMUSER.SKELS)\nAccessing the WSim Test Manager\nuserid.WTMUSER.SKELS   Data set concatenated to the ISPSLIB DD statement (the prefix userid does not need to be specified in the WSIMTM exec for WTMUSER.SKELS) \n   \n Note:  When using the WSim Test Manager, the TSO profile user characteristic, PREFIX, is set to the user ID. This ensures that the user ID is added as the first qualifier for all non-fully qualified data set names. The TSO profile user characteristics, WTPMSG and MSGID, are set to show MVS™ messages and terminal message IDs for debugging purposes. The PREFIX, WTPMSG, and MSGID settings are restored to the original values when the WSim Test Manager ends normally.   The  WSIMTM  exec or CLIST cannot be executed until the new user is set up (as described in  Setting up a new user ). \n         \n Global access to WTM   \n \n Individual access to WTM\n#\n\n\n[Rank 4] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Running WSim with the WSim/ISPF Interface\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=utilities-running-wsim-wsimispf-interface\nScore: 0.047599270939826965\nContent: Running  WSim  with the  WSim/ISPF Interface \n   \n The  WSim/ISPF Interface  is a CUA-compliant operator interface that lets you  start the  WSim  utilities.   To use the  WSim/ISPF Interface , you need to run  WSim  under MVS™ with TSO/E and ISPF.  Refer to  WSim User's Guide  for information about setting up your environment to run the  WSim/ISPF Interface .  \n \n You can access the following  WSim  functions with the  WSim/ISPF Interface :   \n STL Translator \n Preprocessor and ITPSYSIN \n Interactive Data Capture (IDC) Utility \n Log Script Generator Utility \n Script Generator Utility \n SNA 3270 Reformatter Utility \n \n WSim  simulation runs \n Response Time Utility \n Log Compare Utility \n Loglist Utility \n   You can get help for the  WSim/ISPF Interface  at any time by pressing F1. For more information, see  Getting help from the WSim/ISPF Interface .         \n Invoking the WSim/ISPF Interface   \n \n Getting help from the WSim/ISPF Interface   \n \n Navigating the panels   \n \n Entering data on panel fields   \n \n Using function keys\n#\n\n\n[Rank 5] Title: Workload Simulator for z/OS and OS/390 Workload Simulator for z/OS and OS/390 Operation\nURL: https://www.ibm.com/docs/en/wsfz-and-o/1.1.0?topic=guide-operation\nScore: 0.03493978828191757\nContent: Operation           \n Introduction to WSim operation   \n \n Running WSim   \n \n Using operator commands   \n \n Using operator reports   \n \n Controlling message logging   \n \n Using the Display Monitor Facility   \n \n Isolating problems   \n \n Specifying operator commands\n#\n",
                "tool_call_id": "1",
            },
        ]
    )
}

response = generate_answer(input) # type: ignore
last_message = response["messages"][-1]
# print(last_message.to_json())
last_message.pretty_print()

### 7. Assemble the graph

In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode

workflow = StateGraph(MessagesState)

# Define the nodes we will cycle between
workflow.add_node(generate_query_or_respond)
workflow.add_node("retrieve", ToolNode([retriever_tool]))
# workflow.add_node(ToolNode([retriever_tool], name="retrieve"))  # same as above
# workflow.add_node(rewrite_question)  # XXX remove rewrite_question
workflow.add_node(generate_answer)

workflow.add_edge(START, "generate_query_or_respond")
workflow.add_edge("generate_answer", END)
# workflow.add_edge("rewrite_question", "generate_query_or_respond")  # XXX remove rewrite_question

# NOTE: route_on_tool_calls is a path (conditional edge)
# Route based on whether the model requested tool calls.
def route_on_tool_calls(state: MessagesState):
    print_messages(state, "route_on_tool_calls")
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return END

# NOTE: route_on_tool_calls is a path (conditional edge)
# Decide whether to retrieve
workflow.add_conditional_edges(
    "generate_query_or_respond",
    # Assess LLM decision (call `retriever_tool` tool or respond to the user)
    route_on_tool_calls,
    {
        # Translate the condition outputs to nodes in our graph
        "tools": "retrieve",
        END: END,
    },
)

# @@@ahoaho XXX
# # NOTE: grade_documents is a path (conditional edge)
# # Edges taken after the `action` node is called.
# workflow.add_conditional_edges(
#     "retrieve",
#     # Assess agent decision
#     grade_documents,
# )
# XXX remove rewrite_question
workflow.add_edge("retrieve", "generate_answer")

# Compile
graph = workflow.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### 8. Run the agentic RAG

In [ ]:
lg_kwargs = {}

# NOTE uncomment for tracing with langfuse
# from langfuse.langchain import CallbackHandler
# lg_kwargs["config"] = {"callbacks":[CallbackHandler()]}

In [ ]:
input = {
    "messages": convert_to_messages(
        [
            {
                "role": "user",
                "content": "IMSの待機入力（WFI）トランザクションを使用する主な目的は何ですか？",
            }
        ]
    )
}

for chunk in graph.stream(input, **lg_kwargs): # type: ignore
    for node, update in chunk.items():
        print(f"<<< Update from node '{node}' >>>")
        messages = update["messages"]
        last_message = messages[-1]
        last_message.pretty_print()
        print("\n\n")